# QuantHive — Multi-Agent PPO Training
**A Benchmark for Learned Adaptive Governance in Multi-Agent Financial Systems**

This notebook trains all 3 governance agents (Risk Manager, Portfolio Manager, Trader) using PPO.

---

## Step 1: Upload the project
Upload your entire QuantHive project as a `.zip` to Colab, OR clone from GitHub.

In [ ]:
# Option A: Clone from GitHub (recommended if you've pushed)
# !git clone https://github.com/YOUR_USERNAME/QuantHive.git
# %cd QuantHive

# Option B: Upload zip from local machine
from google.colab import files
uploaded = files.upload()  # Upload your QuantHive.zip
!unzip -q QuantHive.zip -d QuantHive
%cd QuantHive

In [ ]:
# Step 2: Install dependencies
!pip install torch gymnasium pettingzoo numpy pandas scipy matplotlib -q

In [ ]:
# Step 3: Verify everything works
!python -m pytest tests/test_regime_engine.py tests/test_governance_logic.py tests/test_environment.py -v --tb=short

In [ ]:
# Step 4: Quick smoke test (5 episodes, should finish in <10 seconds)
import sys
sys.path.insert(0, '.')

from training.ppo_trainer import MultiAgentPPOTrainer

trainer = MultiAgentPPOTrainer(
    max_steps=50,
    output_dir='outputs/smoke_test',
    seed=42,
    device='cpu',
)
trainer.train(total_episodes=5)
print('\nSmoke test passed!')

In [ ]:
# Step 5: FULL TRAINING (this is the real run)
# ~30-60 min on Colab GPU, ~2-4 hours on CPU

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training on: {device}')

trainer = MultiAgentPPOTrainer(
    max_steps=500,
    output_dir='outputs/ppo_training',
    seed=42,
    device=device,
    log_every=10,
    save_every=100,
)
metrics = trainer.train(total_episodes=1500)

In [ ]:
# Step 6: Plot training curves
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

eps = metrics['episode']
kernel = np.ones(20) / 20

for ax, key, title, color in [
    (axes[0,0], 'max_drawdown', 'Max Drawdown', '#e15759'),
    (axes[0,1], 'total_return', 'Total Return', '#4e79a7'),
    (axes[1,0], 'violation_rate', 'Violation Rate', '#f28e2b'),
    (axes[1,1], 'total_failures', 'Governance Failures', '#76b7b2'),
]:
    vals = metrics[key]
    ax.plot(eps, vals, alpha=0.3, color=color, linewidth=0.5)
    if len(vals) > 20:
        smoothed = np.convolve(vals, kernel, mode='valid')
        ax.plot(range(len(smoothed)), smoothed, color=color, linewidth=2)
    ax.set_xlabel('Episode')
    ax.set_title(title)
    ax.grid(True, alpha=0.3)

fig.suptitle('QuantHive — Training Progress', fontsize=14, fontweight='bold')
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('outputs/training_curves.png', dpi=200)
plt.show()

In [ ]:
# Step 7: Run benchmark suite
from evaluation.benchmark_suite import BenchmarkSuite

import glob
# Find best checkpoint
ckpt_dirs = sorted(glob.glob('outputs/ppo_training/best_*'))
best_ckpt = ckpt_dirs[-1] if ckpt_dirs else None
print(f'Using checkpoint: {best_ckpt}')

suite = BenchmarkSuite(
    num_seeds=50,
    max_steps=500,
    checkpoint_dir=best_ckpt,
    output_dir='results/benchmark',
    device=device,
)
suite.run_full()

In [ ]:
# Step 8: Download results
!zip -r results.zip outputs/ results/
from google.colab import files
files.download('results.zip')